In [2]:
pip install requests beautifulsoup4 lxml tqdm

     ---------------------------------------- 73.1/73.1 kB 1.0 MB/s eta 0:00:00
     -------------------------------------- 107.7/107.7 kB 6.1 MB/s eta 0:00:00
     ---------------------------------------- 4.0/4.0 MB 6.8 MB/s eta 0:00:00
     ---------------------------------------- 78.4/78.4 kB 4.5 MB/s eta 0:00:00
     -------------------------------------- 159.3/159.3 kB 3.2 MB/s eta 0:00:00
     ---------------------------------------- 74.2/74.2 kB 4.0 MB/s eta 0:00:00
     -------------------------------------- 131.1/131.1 kB 7.6 MB/s eta 0:00:00
     -------------------------------------- 134.1/134.1 kB 8.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
pip install cloudscraper beautifulsoup4 lxml tqdm

     ---------------------------------------- 99.7/99.7 kB 1.1 MB/s eta 0:00:00
     -------------------------------------- 122.8/122.8 kB 2.4 MB/s eta 0:00:00
     ---------------------------------------- 54.5/54.5 kB 2.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import re
import time
import cloudscraper
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm import tqdm

# ======================
# 설정
# ======================
BASE_URL = "https://www.kongju.ac.kr/KNU/16909/subview.do"

SAVE_DIR = "knu_crawl"
ATTACH_DIR = os.path.join(SAVE_DIR, "attachments")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(ATTACH_DIR, exist_ok=True)

# ======================
# cloudscraper 세션 (핵심)
# ======================
scraper = cloudscraper.create_scraper(
    browser={
        "browser": "chrome",
        "platform": "windows",
        "mobile": False
    }
)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8"
}

# ======================
# 유틸
# ======================
def get_soup(url):
    res = scraper.get(url, headers=HEADERS, timeout=15)
    res.raise_for_status()
    return BeautifulSoup(res.text, "lxml")


def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()


def make_filename(title):
    return re.sub(r"[^a-zA-Z0-9가-힣]", "_", title)[:80]


# ======================
# 페이지 URL 생성
# ======================
def make_page_url(base_url, page):
    if "pageIndex=" in base_url:
        return re.sub(r"pageIndex=\d+", f"pageIndex={page}", base_url)
    else:
        sep = "&" if "?" in base_url else "?"
        return f"{base_url}{sep}pageIndex={page}"


# ======================
# 게시글 링크 수집 (강화 버전)
# ======================
def get_post_links(list_url):
    soup = get_soup(list_url)

    links = set()

    for a in soup.select("a[href]"):
        href = a["href"]

        if any(x in href for x in ["artclView", "articleView", "view.do"]):
            links.add(urljoin(list_url, href))

    for a in soup.select("a[onclick]"):
        onclick = a.get("onclick", "")
        match = re.findall(r"'(.*?)'", onclick)
        for m in match:
            if "view" in m or "artcl" in m:
                links.add(urljoin(list_url, m))

    return list(links)


# ======================
# 본문 추출 (iframe + fallback)
# ======================
def extract_content(soup):
    selectors = [
        ".board_view",
        ".board-view",
        ".board_view_content",
        ".view-content",
        ".view_detail",
        ".tb_contents",
        ".dbData",
        ".content",
        "#contents"
    ]

    for sel in selectors:
        tag = soup.select_one(sel)
        if tag:
            text = tag.get_text("\n", strip=True)
            if len(text) > 30:
                return text

    # iframe 대응
    iframe = soup.select_one("iframe")
    if iframe and iframe.get("src"):
        iframe_url = urljoin(BASE_URL, iframe["src"])
        iframe_soup = get_soup(iframe_url)
        return iframe_soup.get_text("\n", strip=True)

    return ""


# ======================
# 첨부파일 추출 (K2Web 대응)
# ======================
def extract_files(soup, base_url):
    files = set()

    # 1. href 기반
    for a in soup.select("a[href]"):
        href = a["href"]
        if any(ext in href.lower() for ext in [".hwp", ".hwpx", ".pdf", ".xls", ".xlsx", ".zip"]):
            files.add(urljoin(base_url, href))

    # 2. onclick 기반
    for a in soup.select("a[onclick]"):
        onclick = a.get("onclick", "")

        if "download" in onclick.lower() or "file" in onclick.lower():
            matches = re.findall(r"'(.*?)'", onclick)
            for m in matches:
                files.add(urljoin(base_url, m))

    return list(files)


# ======================
# 다운로드
# ======================
def download_file(file_url):
    try:
        filename = os.path.basename(urlparse(file_url).path)
        filepath = os.path.join(ATTACH_DIR, filename)

        r = scraper.get(file_url, stream=True, headers=HEADERS, timeout=20)
        r.raise_for_status()

        with open(filepath, "wb") as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

        print(f"[다운로드 완료] {filename}")

    except Exception as e:
        print(f"[다운로드 실패] {file_url} -> {e}")


# ======================
# 게시글 파싱
# ======================
def parse_post(url):
    soup = get_soup(url)

    title_tag = soup.select_one("h2, .board_view_tit, .title")
    title = clean_text(title_tag.get_text()) if title_tag else "No Title"

    content = extract_content(soup)
    file_links = extract_files(soup, url)

    filename = make_filename(title)

    with open(os.path.join(SAVE_DIR, f"{filename}.txt"), "w", encoding="utf-8") as f:
        f.write(title + "\n\n")
        f.write(content + "\n\n")
        f.write("첨부파일:\n")
        f.write("\n".join(file_links))

    for file_url in file_links:
        download_file(file_url)

    return {
        "title": title,
        "content": content,
        "files": file_links,
        "url": url
    }


# ======================
# 크롤러
# ======================
def crawl_board(start_url, max_pages=5, delay=0.5):
    results = []
    visited = set()

    for page in range(1, max_pages + 1):
        list_url = make_page_url(start_url, page)

        print(f"\n[LIST PAGE] {list_url}")

        try:
            post_links = get_post_links(list_url)
        except Exception as e:
            print(f"[LIST ERROR] {e}")
            continue

        for post_url in tqdm(post_links):
            if post_url in visited:
                continue

            visited.add(post_url)

            try:
                print(f"  └─ post: {post_url}")
                data = parse_post(post_url)
                results.append(data)

                time.sleep(delay)

            except Exception as e:
                print(f"[POST ERROR] {post_url} -> {e}")

    return results


# ======================
# 실행
# ======================
if __name__ == "__main__":
    data = crawl_board(BASE_URL, max_pages=3)
    print(f"\n총 수집 게시글: {len(data)}")


# 지금 상황은 “requests 계열로는 구조적으로 안 나오는 사이트”라서 Selenium으로 정리하는 게 맞는 방향이야.
# 이번에는 공주대 K2Web 기준으로 실제 동작 가능한 전체 통합 Selenium 크롤러로 다시 짜줄게.


[LIST PAGE] https://www.kongju.ac.kr/KNU/16909/subview.do?pageIndex=1


  0%|          | 0/138 [00:00<?, ?it/s]

  └─ post: https://www.kongju.ac.kr/bbs/KNU/2132/419865/artclView.do


  1%|          | 1/138 [00:01<02:17,  1.00s/it]

  └─ post: https://www.kongju.ac.kr/KNU/16568/subview.do


...


[LIST PAGE] https://www.kongju.ac.kr/KNU/16909/subview.do?pageIndex=2


100%|██████████| 138/138 [00:00<?, ?it/s]



[LIST PAGE] https://www.kongju.ac.kr/KNU/16909/subview.do?pageIndex=3


100%|██████████| 138/138 [00:00<00:00, 136997.39it/s]


총 수집 게시글: 138
